In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import numpy as np


def _parse(response, a, b):
    text = str(response).strip().lower()
    for token, opt in [(a.lower(), a), (b.lower(), b)]:
        if re.search(r"\b" + re.escape(token) + r"\b", text):
            return opt

    pos_a, pos_b = text.find(a.lower()), text.find(b.lower())
    if pos_a >= 0 and pos_b >= 0:
        return a if pos_a < pos_b else b
    if pos_a >= 0:
        return a
    if pos_b >= 0:
        return b
    return None


def _slope(x, y):
    if len(x) < 2:
        return 0.0
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    xm, ym = x.mean(), y.mean()
    denom = ((x - xm) ** 2).sum()
    return float(((x - xm) * (y - ym)).sum() / denom) if denom else 0.0


# N_TRAIN_BLK = 3
# N_TEST_BLK = 3
# N_SESSIONS = 3
# HISTORY_LEN = 20
N_TRAIN_BLK = 1
N_TEST_BLK = 1
N_SESSIONS = 1
HISTORY_LEN = 10

def _run_session(llm):
    base = [f"Stim-{chr(65+i)}" for i in range(10)]
    random.shuffle(base)

    pool_L = base[:5]
    pool_U = base[5:]

    adj_L = [(pool_L[i], pool_L[i + 1]) for i in range(4)]
    adj_U = [(pool_U[i], pool_U[i + 1]) for i in range(4)]

    all_L = [(pool_L[i], pool_L[j]) for i in range(5) for j in range(i + 1, 5)]
    all_U = [(pool_U[i], pool_U[j]) for i in range(5) for j in range(i + 1, 5)]

    trials = []

    for _ in range(N_TRAIN_BLK):
        blk = []
        for a, b in adj_L:
            blk += [
                {"is_L": True, "test": False, "a": a, "b": b},
                {"is_L": True, "test": False, "a": b, "b": a},
            ]
        for a, b in adj_U:
            blk += [
                {"is_L": False, "test": False, "a": a, "b": b},
                {"is_L": False, "test": False, "a": b, "b": a},
            ]
        random.shuffle(blk)
        trials.extend(blk)

    for _ in range(N_TEST_BLK):
        blk = []
        for a, b in all_L:
            blk += [
                {"is_L": True, "test": True, "a": a, "b": b},
                {"is_L": True, "test": True, "a": b, "b": a},
            ]
        for a, b in all_U:
            blk += [
                {"is_L": False, "test": True, "a": a, "b": b},
                {"is_L": False, "test": True, "a": b, "b": a},
            ]
        random.shuffle(blk)
        trials.extend(blk)

    llm.prompt(
        "LEARNING TASK\n"
        f"Group 1: {', '.join(pool_L)}\n"
        f"Group 2: {', '.join(pool_U)}\n"
        "Some patterns may be more learnable than others. "
        "Use the feedback history to maximize reward over time. "
        "Reply with ONLY the stimulus name."
    )

    L_outcomes = []
    history = []

    choices_L = {item: 0 for item in pool_L}
    appeared_L = {item: 0 for item in pool_L}
    choices_U = {item: 0 for item in pool_U}
    appeared_U = {item: 0 for item in pool_U}

    invalid_count = 0

    for t_num, trial in enumerate(trials):
        a, b = trial["a"], trial["b"]
        is_L = trial["is_L"]
        is_test = trial["test"]

        recent = history[-HISTORY_LEN:]
        history_text = "\n".join(recent) if recent else "No previous trials."

        resp = llm.prompt(
            f"Recent history:\n{history_text}\n\n"
            f"Trial {t_num + 1}: {a} vs {b}\n"
            f"Reply with ONLY the stimulus name."
        )

        choice = _parse(resp, a, b)
        if choice is None:
            invalid_count += 1
            choice = random.choice([a, b])

        if is_L:
            ra, rb = pool_L.index(a), pool_L.index(b)
            target = a if ra < rb else b
            rewarded = int(choice == target)
            L_outcomes.append(rewarded)
        else:
            prob = float(np.mean(L_outcomes[-10:])) if L_outcomes else 0.5
            rewarded = int(random.random() < prob)

        history.append(
            f"Trial {t_num + 1}: {a} vs {b} | chose {choice} | "
            f"{'reward' if rewarded else 'no reward'}"
        )

        if is_test:
            if is_L:
                appeared_L[a] += 1
                appeared_L[b] += 1
                choices_L[choice] += 1
            else:
                appeared_U[a] += 1
                appeared_U[b] += 1
                choices_U[choice] += 1

    cf_L = [
        choices_L[item] / appeared_L[item] if appeared_L[item] > 0 else 0.5
        for item in pool_L
    ]
    cf_U = [
        choices_U[item] / appeared_U[item] if appeared_U[item] > 0 else 0.5
        for item in pool_U
    ]

    op_L = _slope([1, 2, 3, 4, 5], cf_L)
    op_U = _slope([1, 2, 3, 4, 5], cf_U)
    op_disc = op_L - op_U

    return {
        "op_L": op_L,
        "op_U": op_U,
        "op_disc": op_disc,
        "invalid_rate": invalid_count / len(trials),
    }


@kbench.task(
    name="Learnability (PN) - Ordering Preference Discrimination Eval",
    description=(
        "Two interleaved 5-item stimulus sets. One set has a consistent hidden "
        "ranking; the other uses a PN reward schedule yoked to recent reward rate. "
        "Returns OP_L, OP_U, OP_discrimination, and invalid response rate."
    ),
)
def learnability_pn_op_discrimination_eval(llm, seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)

    session_metrics = [_run_session(llm) for _ in range(N_SESSIONS)]

    mean_op_L = float(np.mean([m["op_L"] for m in session_metrics]))
    mean_op_U = float(np.mean([m["op_U"] for m in session_metrics]))
    mean_op_disc = float(np.mean([m["op_disc"] for m in session_metrics]))
    mean_invalid = float(np.mean([m["invalid_rate"] for m in session_metrics]))

    return {
        "seed": seed,
        "op_L": mean_op_L,
        "op_U": mean_op_U,
        "op_disc": mean_op_disc,
        "invalid_rate": mean_invalid,
    }



In [ ]:

# evaluation_df = pd.DataFrame({"seed": list(range(2))})

# models = [
#     kbench.llms["google/gemini-2.0-flash"],
#     kbench.llms["google/gemini-2.5-flash"],
#     kbench.llms["google/gemini-2.5-pro"],
# ]

# results = learnability_pn_op_discrimination_eval.evaluate(
#     llm=models,
#     evaluation_data=evaluation_df,
# )


evaluation_df = pd.DataFrame({"seed": [0]})


models = [
    kbench.llms["google/gemini-2.5-flash"],
    kbench.llms["google/gemini-2.5-pro"],
    kbench.llms["openai/gpt-5.4-mini-2026-03-17"],
    kbench.llms["openai/gpt-5.4-2026-03-05"],
    kbench.llms["anthropic/claude-haiku-4-5@20251001"],
    kbench.llms["anthropic/claude-opus-4-6@default"],
    kbench.llms["deepseek-ai/deepseek-r1-0528"],
]

results = learnability_pn_op_discrimination_eval.evaluate(
    llm=models,
    evaluation_data=evaluation_df,
)


In [ ]:

# df = results.as_dataframe()
# print(df)
# print(df.columns)

# model_col = "model" if "model" in df.columns else [c for c in df.columns if "model" in c.lower()][0]
# print(df.groupby(model_col)[["op_L", "op_U", "op_disc", "invalid_rate"]].mean())

print(df.columns.tolist())
print(df.head(3).to_dict(orient="records"))

In [ ]:
df = results.as_dataframe().copy()

# model name from llm object
df["model"] = df["llm"].apply(lambda x: getattr(x, "name", str(x)))

# preserve the SAME index as df
metrics = pd.DataFrame(df["result"].tolist(), index=df.index)

# avoid duplicate seed column
metrics = metrics.drop(columns=["seed"], errors="ignore")

# combine cleanly
df2 = pd.concat([df.drop(columns=["result"]), metrics], axis=1)

print(df2.columns.tolist())
print(df2.head())

summary = (
    df2.groupby("model")[["op_L", "op_U", "op_disc", "invalid_rate"]]
    .mean()
    .sort_values("op_disc")
)

print(summary)